In [18]:
import numpy as np

In [19]:
class Layer:
    def __init__(self):
        self.input = None
        self.output = None
    def forward(self,inputs):
        pass
    def backward(self,outputGradient,learningRate):
        pass

In [52]:
class Dense(Layer):
    def __init__(self,inputSize,outputSize):
        np.random.seed(42)
        self.weights = np.random.randn(inputSize,outputSize) * 0.01
        self.bias = np.zeros((1,outputSize))
    def forward(self,inputs):
        self.input = inputs
        return np.dot(self.input,self.weights) + self.bias
    def backward(self,outputGradient,learningRate):
        weightsGradient = np.dot(self.input.T,outputGradient)
        biasGradient = np.sum(outputGradient, axis = 0,keepdims = True)
        weightChange = np.dot(outputGradient,self.weights.T)
        self.weights = self.weights + (learningRate * weightsGradient)
        self.bias = self.bias + (learningRate * biasGradient)
        return weightChange

In [53]:
class ReLU(Layer):
    def forward(self,inputs):
        self.input = inputs
        return np.maximum(0,self.input)
    def backward(self,outputGradient,learningRate):
        derivative = (self.input > 0)
        return outputGradient * derivative

In [54]:
class Sigmoid(Layer):
    def forward(self,inputs):
        self.output = 1/(1+np.exp(-inputs))
        return self.output
    def backward(self,outputGradient,learningRate):
        derivative = self.output * (1-self.output)
        return outputGradient * derivative
        

In [55]:
def mseLoss(yTrue,yPred):
    return np.mean(np.power(yTrue-yPred,2))
def mseLossGradient(yTrue,yPred):
    return (2*(yTrue-yPred))/yTrue.size

In [56]:
class NeuralNetwork:
    def __init__(self):
        self.layers = list()
    def add(self,layer):
        self.layers.append(layer)
    def predict(self,X):
        output = X
        for layer in self.layers:
            output = layer.forward(output)
        return output
    def train(self,X,yTrue,epochs,lr):
        for epoch in range(epochs):
            predictions = self.predict(X)
            error = mseLoss(yTrue,predictions)
            grad = mseLossGradient(yTrue,predictions)
            for layer in reversed(self.layers):
                grad  = layer.backward(grad, lr)
            
    

In [57]:
X_train = np.array([
        [0, 0],
        [0, 1],
        [1, 0],
        [1, 1]
    ])
    
Y_train = np.array([
    [0],
    [1],
    [1],
    [0]
])

# Build the network
model = NeuralNetwork()

# Hidden Layer: 2 inputs -> 4 neurons
model.add(Dense(2, 4))
model.add(ReLU())

# Output Layer: 4 neurons -> 1 final output
model.add(Dense(4, 1))
model.add(Sigmoid())

# Train the network
print("Training started...")
model.train(X_train, Y_train, epochs=10000, lr=0.5)

# Test the final predictions
print("\nTraining complete. Final Predictions:")
predictions = model.predict(X_train)

for i in range(len(X_train)):
    print(f"Input: {X_train[i]} | True: {Y_train[i][0]} | Predicted: {predictions[i][0]:.4f}")

Training started...

Training complete. Final Predictions:
Input: [0 0] | True: 0 | Predicted: 0.0122
Input: [0 1] | True: 1 | Predicted: 0.9941
Input: [1 0] | True: 1 | Predicted: 0.9941
Input: [1 1] | True: 0 | Predicted: 0.0122
